# Epic of Sun - Desafio 1
Benchmarking the models
***

This jupyter notebook benchmarks the models

It is divided in 2 parts:
* Loading libraries
* Benchmarking

***

## Loading libraries

In [16]:
import time
import numpy as np

import tensorflow as tf
from keras.models import load_model

from pathlib import Path
import json

## Benchmarking

In [17]:
# Select the type of model you want to analyze. You can choose between "unet", "fpn", and "attention_unet".
# The code will load the corresponding model and its training history based on your selection.
flag_model = 5

if flag_model == 1:
    model_type = "unet"
    model_type_name = "U-Net"
elif flag_model == 2:
    model_type = "fpn"
    model_type_name = "FPN"
elif flag_model == 3:
    model_type = "attention_unet"
    model_type_name = "Attention U-Net"
elif flag_model == 4:
    model_type = "deeplabv3"
    model_type_name = "DeepLabV3"
elif flag_model == 5:
    model_type = "deeplabv3_plus"
    model_type_name = "DeepLabV3+"

In [18]:
# Get the directory where THIS script is located
base_path = Path.cwd()
base_path = base_path.parent  # Move one level up to the project root

In [19]:
# Function to load the model from the specified path

def load_model_from_path(model_path):
    if model_path.exists():
        try:
            # Pass str(model_path) to ensure compatibility
            model = load_model(str(model_path), compile=False)
            file_name = model_path.name
            print(f"Successfully loaded {file_name} model.")
            return model
        except Exception as e:
            print(f"File {file_name} exists, but load_model failed with error:\n{e}")
    else:
        print(f"File {file_name} does not exist at the specified path. Check your working directory or base_path.")

In [20]:
# Function to benchmark the model

def benchmark_model(model, input_shape=(1, 128, 128, 7), n_warmup=10, n_runs=100):
    """
    Benchmarks a trained segmentation model for inference cost.
    input_shape: (batch, H, W, C) -- use batch=1 to simulate single-patch
    edge inference, which is the realistic deployment scenario.
    """
    dummy_input = tf.random.uniform(input_shape)

    # Warm-up
    for _ in range(n_warmup):
        _ = model(dummy_input, training=False).numpy()  # .numpy() forces sync here too

    latencies = []
    for _ in range(n_runs):
        start = time.perf_counter()
        output = model(dummy_input, training=False)
        _ = output.numpy()          # <-- blocks until the GPU actually finishes
        latencies.append(time.perf_counter() - start)

    latencies = np.array(latencies) * 1000  # ms

    n_params = model.count_params()
    # Use model.weights (all weights), not just trainable_weights --
    # BatchNorm's moving_mean/moving_variance are non-trainable but still
    # occupy memory and are part of the saved model, so excluding them
    # under-counts size (more so for models with many BN layers, like this one).
    param_bytes = sum(
        np.prod(w.shape) * np.dtype(str(w.dtype)).itemsize
        for w in model.weights
    )

    results = {
        "n_params": n_params,
        "param_size_mb": param_bytes / (1024 ** 2),
        "latency_mean_ms": latencies.mean(),
        "latency_p50_ms": np.percentile(latencies, 50),
        "latency_p95_ms": np.percentile(latencies, 95),
        "latency_std_ms": latencies.std(),
    }
    return results

In [21]:
# Function to print the results in a readable format
def print_benchmark_results(results, model_name="Model"):
    print(f"Benchmark results for {model_name}:")
    print(f"Number of parameters: {results['n_params']}")
    print(f"Parameter size (MB): {results['param_size_mb']:.2f}")
    print(f"Mean latency (ms): {results['latency_mean_ms']:.2f}")
    print(f"Median latency (ms): {results['latency_p50_ms']:.2f}")
    print(f"95th percentile latency (ms): {results['latency_p95_ms']:.2f}")
    print(f"Latency standard deviation (ms): {results['latency_std_ms']:.2f}")

    # Save benchmark data as a json file for future reference
    metrics_data = {
        "Model": model_name,
        "Number of parameters": results['n_params'],
        "Parameter size (MB)": results['param_size_mb'],
        "Mean latency (ms)": results['latency_mean_ms'],
        "Median latency (ms)": results['latency_p50_ms'],
        "95th percentile latency (ms)": results['latency_p95_ms'],
        "Latency standard deviation (ms)": results['latency_std_ms'],
    }

    metrics_path = base_path / "models" / f"{model_type}" / f"{model_type}_benchmarks_data.json"
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    with open(str(metrics_path), "w") as f:
        json.dump(metrics_data, f)
    print(f"Benchmark data saved to '{metrics_path}'")

In [22]:
# Load the models
model = load_model_from_path(base_path / "models" / f"{model_type}" / f"{model_type}_sugarcane.keras")

Successfully loaded deeplabv3_plus_sugarcane.keras model.


In [23]:
# Usage per architecture:
results = benchmark_model(model)

In [24]:
# Print the results
print_benchmark_results(results, model_name=model_type_name)

Benchmark results for DeepLabV3+:
Number of parameters: 26744961
Parameter size (MB): 102.02
Mean latency (ms): 146.97
Median latency (ms): 145.03
95th percentile latency (ms): 158.73
Latency standard deviation (ms): 5.75
Benchmark data saved to '/home/fdesmello/Git/spacesugar2/models/deeplabv3_plus/deeplabv3_plus_benchmarks_data.json'
